# Imports

In [1]:
#============================================================

#Imports

#============================================================

import os 
import gc 
import json
import random
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import scvi

import torch
import torch.nn as nn 
import torch.optim as optim

from scipy import sparse
from sklearn.model_selection import train_test_split



/home/users/vvishakha/miniconda3/envs/scvi_env_copy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Configuration

In [ ]:

# ============================================================

# Configuration

# ============================================================

class Config:

    # -------------------------
    # Paths
    # -------------------------

    base_dir = "Path/to_base_directory"

    data_dir = os.path.join(
        base_dir,
        "data"
    )

    model_dir = os.path.join(
        base_dir,
        
        "best_models_v2"
    )

    output_dir = "./Output_directory"

    os.makedirs(output_dir, exist_ok=True)

    sn_data = os.path.join(
        data_dir,
        "sn_data.h5ad"
    )

    sc_data = os.path.join(
        data_dir,
        "sc_data.h5ad"
    )

    sn_model = os.path.join(
        model_dir,
        "nucleus_raw_BEST",
        "model"
    )

    sc_model = os.path.join(
        model_dir,
        "cell_raw_BEST",
        "model"
    )

    # -------------------------
    # Latent Space
    # -------------------------

    n_latent = 30
    n_genes = 17520

    # -------------------------
    # Conditional GAN
    # -------------------------

    embedding_dim = 32
    gan_hidden = 512
    gan_res_blocks = 4

    # -------------------------
    # Training
    # -------------------------

    batch_size = 1024
    gan_epochs = 200

    lr = 1e-4

    n_critic = 5

    lambda_cycle = 10.0
    lambda_id = 5.0
    lambda_gp = 10.0
    lambda_cell = 2.0
    lambda_centroid = 1.0

    fld_eval_every = 1

    balanced_sampling = True

    decode_chunk = 20000

    seed = 42

    device = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Config()

device = torch.device(cfg.device)

# ============================================================

# Reproducibility

# ============================================================

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

if torch.cuda.is_available(): torch.cuda.manual_seed_all(cfg.seed)
print("=" * 60)
print("CycleGAN")
print("=" * 60)
print(f"Device :{device}")
print(f"scvi-tools : {scvi.__version__}")
print(f"Torch : {torch.__version__}")
print(f"Output Dir : {cfg.output_dir}")
print("=" * 60)

# Helper Functions

In [ ]:
# ————————————————————

# Helper Functions

# ————————————————————

def extract_latents(domain_name, data_path, model_path, prefix):

    latent_file = os.path.join(cfg.output_dir, f"{prefix}_latents.npz")

    if os.path.exists(latent_file):

        print(f"[{domain_name}] Loading cached latents...")

        data = np.load(latent_file, allow_pickle=True)

        return data

    print(f"[{domain_name}] Reading AnnData...")
    adata = sc.read_h5ad(data_path)

    for col in ["cell_type", "dataset_id"]:
        if col in adata.obs:
            adata.obs[col] = adata.obs[col].astype("category")

    print(f"[{domain_name}] Loading SCVI model...")
    model = scvi.model.SCVI.load(model_path, adata=adata)

    print(f"[{domain_name}] Extracting latent representation...")
    z = model.get_latent_representation()

    X = adata.X

    if sparse.issparse(X):
        library = np.asarray(X.sum(axis=1)).ravel()
    else:
        library = X.sum(axis=1).ravel()

    log_library = np.log1p(library).astype(np.float32)

    batch_idx = adata.obs["_scvi_batch"].values.astype(np.int64)

    np.savez_compressed(
        latent_file,
        z=z,
        log_library=log_library,
        batch_idx=batch_idx,
        cell_type=adata.obs["cell_type"].astype(str).values,
        dataset_id=adata.obs["dataset_id"].astype(str).values
    )

    batch_registry = model.adata_manager.get_state_registry(
        scvi.REGISTRY_KEYS.BATCH_KEY
    )

    with open(
        os.path.join(cfg.output_dir, f"{prefix}_batch_mapping.json"),
        "w"
    ) as f:

        json.dump(
            dict(enumerate(batch_registry.categorical_mapping)),
            f,
            indent=2
        )

    del model, adata, X
    gc.collect()
    torch.cuda.empty_cache()

    return np.load(latent_file, allow_pickle=True)

# ————————————————————

# SN

# ————————————————————

sn = extract_latents( "SN", cfg.sn_data, cfg.sn_model, "sn" )

z_sn = sn["z"] 
sn_cell_types = sn["cell_type"]
sn_batch_idx = sn["batch_idx"] 
sn_dataset_ids = sn["dataset_id"] 
log_lib_sn =sn["log_library"]

print("SN:", z_sn.shape)

# ————————————————————

# SC

# ————————————————————

sc_data_npz = extract_latents( "SC", cfg.sc_data, cfg.sc_model, "sc" )

z_sc = sc_data_npz["z"]
sc_cell_types = sc_data_npz["cell_type"]
sc_batch_idx = sc_data_npz["batch_idx"]
sc_dataset_ids = sc_data_npz["dataset_id"]
log_lib_sc = sc_data_npz["log_library"]

print("SC:", z_sc.shape)

print("\nLatents ready.")


# Data Loaders

In [ ]:
# ============================================================

# Metadata & Shared Cell-Type Mapping

# ============================================================



metadata_file = os.path.join(cfg.output_dir, "shared_metadata.npz")

if os.path.exists(metadata_file):

    print("Loading cached metadata...")

    meta = np.load(metadata_file, allow_pickle=True)

    celltype_to_id = meta["celltype_to_id"].item()
    id_to_celltype = meta["id_to_celltype"].item()

    sn_ct = meta["sn_ct"]
    sc_ct = meta["sc_ct"]

    sn_centroids = meta["sn_centroids"]
    sc_centroids = meta["sc_centroids"]

    sn_weights = meta["sn_weights"]
    sc_weights = meta["sc_weights"]

    sn_ct_library = meta["sn_ct_library"]
    sc_ct_library = meta["sc_ct_library"]

    sn_ct_batch = meta["sn_ct_batch"]
    sc_ct_batch = meta["sc_ct_batch"]

else:

    print("Building metadata...")

    shared = sorted(
        set(np.unique(sn_cell_types))
        &
        set(np.unique(sc_cell_types))
    )

    celltype_to_id = {
        ct:i for i,ct in enumerate(shared)
    }

    id_to_celltype = {
        i:ct for ct,i in celltype_to_id.items()
    }

    sn_ct = np.array([celltype_to_id[x] for x in sn_cell_types],dtype=np.int64)
    sc_ct = np.array([celltype_to_id[x] for x in sc_cell_types],dtype=np.int64)

    n_types = len(shared)

    sn_centroids = np.zeros((n_types,cfg.n_latent),dtype=np.float32)
    sc_centroids = np.zeros((n_types,cfg.n_latent),dtype=np.float32)

    sn_weights = np.zeros(n_types,dtype=np.float32)
    sc_weights = np.zeros(n_types,dtype=np.float32)

    sn_ct_library = np.zeros(n_types,dtype=np.float32)
    sc_ct_library = np.zeros(n_types,dtype=np.float32)

    sn_ct_batch = np.zeros(n_types,dtype=np.int64)
    sc_ct_batch = np.zeros(n_types,dtype=np.int64)

    beta = 0.9999

    for ct,idx in celltype_to_id.items():

        sm = sn_cell_types==ct
        cm = sc_cell_types==ct

        sn_centroids[idx]=z_sn[sm].mean(0)
        sc_centroids[idx]=z_sc[cm].mean(0)

        ns = sm.sum()
        nc = cm.sum()

        sn_weights[idx]=(1-beta)/(1-beta**max(ns,1))
        sc_weights[idx]=(1-beta)/(1-beta**max(nc,1))

        sn_ct_library[idx]=np.median(log_lib_sn[sm])
        sc_ct_library[idx]=np.median(log_lib_sc[cm])

        sn_ct_batch[idx]=np.bincount(sn_batch_idx[sm]).argmax()
        sc_ct_batch[idx]=np.bincount(sc_batch_idx[cm]).argmax()

    sn_weights/=sn_weights.mean()
    sc_weights/=sc_weights.mean()

    np.savez_compressed(
        metadata_file,
        celltype_to_id=celltype_to_id,
        id_to_celltype=id_to_celltype,
        sn_ct=sn_ct,
        sc_ct=sc_ct,
        sn_centroids=sn_centroids,
        sc_centroids=sc_centroids,
        sn_weights=sn_weights,
        sc_weights=sc_weights,
        sn_ct_library=sn_ct_library,
        sc_ct_library=sc_ct_library,
        sn_ct_batch=sn_ct_batch,
        sc_ct_batch=sc_ct_batch
    )

split_file=os.path.join(cfg.output_dir,"train_test_split.npz")

if os.path.exists(split_file):

    split=np.load(split_file)

    idx_sn_train=split["idx_sn_train"]
    idx_sn_test=split["idx_sn_test"]

    idx_sc_train=split["idx_sc_train"]
    idx_sc_test=split["idx_sc_test"]

else:

    idx_sn_train,idx_sn_test=train_test_split(
        np.arange(len(z_sn)),
        test_size=0.2,
        random_state=cfg.seed,
        stratify=sn_ct
    )

    idx_sc_train,idx_sc_test=train_test_split(
        np.arange(len(z_sc)),
        test_size=0.2,
        random_state=cfg.seed,
        stratify=sc_ct
    )

    np.savez(
        split_file,
        idx_sn_train=idx_sn_train,
        idx_sn_test=idx_sn_test,
        idx_sc_train=idx_sc_train,
        idx_sc_test=idx_sc_test
    )

print(f"Shared cell types: {len(celltype_to_id)}")
print("Metadata ready.")


# Latent Space Diagnostics

In [ ]:
# ============================================================

# Latent Space Diagnostics

# ============================================================

import os
import json
import numpy as np
import matplotlib.pyplot as plt

from scipy.linalg import orthogonal_procrustes 
from sklearn.cross_decomposition import CCA
from sklearn.linear_model import Ridge

print("=" * 60)
print("Latent Space Diagnostics")
print("=" * 60)

# ————————————————————

# Shared centroids

# ————————————————————

C_sn = sn_centroids
C_sc = sc_centroids

# ————————————————————

# Procrustes

# ————————————————————

A = C_sn - C_sn.mean(0)
B = C_sc - C_sc.mean(0)

A /= np.linalg.norm(A)
B /= np.linalg.norm(B)

R, scale = orthogonal_procrustes(A, B)

aligned = A @ R

procrustes = np.sum((aligned - B) ** 2)

# ————————————————————

# CCA

# ————————————————————

n_components = min( 10, cfg.n_latent, len(celltype_to_id)-1 )

cca = CCA( n_components=n_components, max_iter=1000 )

cca.fit(C_sn, C_sc)

Xc, Yc = cca.transform(C_sn, C_sc)

cca_corr = [ np.corrcoef( Xc[:,i], Yc[:,i] )[0,1] for i in
range(n_components)]

# ————————————————————

# Ridge

# ————————————————————

ridge = Ridge(alpha=1.0)

ridge.fit(C_sn, C_sc)

pred = ridge.predict(C_sn)

r2 = []

for d in range(cfg.n_latent):

    ss_res = np.sum((C_sc[:,d]-pred[:,d])**2)
    ss_tot = np.sum((C_sc[:,d]-C_sc[:,d].mean())**2)

    r2.append(
        1-ss_res/(ss_tot+1e-10)
    )

r2=np.asarray(r2)

# ————————————————————

# Plot

# ————————————————————

fig,ax=plt.subplots(1,3,figsize=(15,4))

ax[0].bar(range(n_components),cca_corr) 
ax[0].set_title("CCA")

ax[1].bar(range(cfg.n_latent),r2)
ax[1].set_title("Ridge R²")

ax[2].scatter(Xc[:,0],Yc[:,0])
m=max(abs(Xc[:,0]).max(),abs(Yc[:,0]).max())
ax[2].plot([-m,m],[-m,m],"k")
ax[2].set_title("CCA Component 1")

plt.tight_layout()

plt.savefig( os.path.join( cfg.output_dir, "latent_diagnostics.png" ),
dpi=200 )

plt.show()

diag={

    "procrustes":float(procrustes),

    "cca":cca_corr,

    "ridge_r2":r2.tolist()

}

with open( os.path.join( cfg.output_dir, "latent_diagnostics.json" ),"w" ) as f:

    json.dump(diag,f,indent=2)

print(f"Procrustes : {procrustes:.4f}")
print(f"Mean Ridge R² : {r2.mean():.4f}")
print("Diagnostics saved.")
# ============================================================
# Move metadata to GPU
# ============================================================

sn_centroids = torch.as_tensor(
    sn_centroids,
    dtype=torch.float32,
    device=device
)

sc_centroids = torch.as_tensor(
    sc_centroids,
    dtype=torch.float32,
    device=device
)

sn_weights = torch.as_tensor(
    sn_weights,
    dtype=torch.float32,
    device=device
)

sc_weights = torch.as_tensor(
    sc_weights,
    dtype=torch.float32,
    device=device
)

print(type(sn_centroids), sn_centroids.device)
print(type(sc_centroids), sc_centroids.device)
print(type(sn_weights), sn_weights.device)
print(type(sc_weights), sc_weights.device)

#  Model Train

In [ ]:
# ============================================================

# Cell 5 — Dataset, Sampler & DataLoader

# ============================================================

import torch
from torch.utils.data  import Dataset, DataLoader, WeightedRandomSampler 
from itertools import cycle

# ————————————————————

# Dataset

# ————————————————————

class CellTypeDataset(Dataset):

    def __init__(self, latent, celltype):

        self.latent = torch.tensor(
            latent,
            dtype=torch.float32
        )

        self.celltype = torch.tensor(
            celltype,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.latent)

    def __getitem__(self, idx):

        return {
            "latent": self.latent[idx],
            "celltype": self.celltype[idx]
        }

# ————————————————————

# Train/Test Arrays

# ————————————————————

z_sn_train = z_sn[idx_sn_train] 
z_sn_test_np = z_sn[idx_sn_test]

z_sc_train = z_sc[idx_sc_train] 
z_sc_test_np = z_sc[idx_sc_test]

sn_ct_train = sn_ct[idx_sn_train] 
sn_ct_test = sn_ct[idx_sn_test]

sc_ct_train = sc_ct[idx_sc_train] 
sc_ct_test = sc_ct[idx_sc_test]

# ————————————————————

# Datasets

# ————————————————————

sn_train = CellTypeDataset( z_sn_train, sn_ct_train )

sc_train = CellTypeDataset( z_sc_train, sc_ct_train )

# ————————————————————

# Weighted Samplers

# ————————————————————

sn_sample_weights = torch.tensor( sn_weights[sn_ct_train],
dtype=torch.double )

sc_sample_weights = torch.tensor( sc_weights[sc_ct_train],
dtype=torch.double )

if cfg.balanced_sampling:

    sn_sampler = WeightedRandomSampler(
        weights=sn_sample_weights,
        num_samples=len(sn_sample_weights),
        replacement=True
    )

    sc_sampler = WeightedRandomSampler(
        weights=sc_sample_weights,
        num_samples=len(sc_sample_weights),
        replacement=True
    )

    sn_loader = DataLoader(
        sn_train,
        batch_size=cfg.batch_size,
        sampler=sn_sampler,
        drop_last=True,
        pin_memory=True
    )

    sc_loader = DataLoader(
        sc_train,
        batch_size=cfg.batch_size,
        sampler=sc_sampler,
        drop_last=True,
        pin_memory=True
    )

else:

    sn_loader = DataLoader(
        sn_train,
        batch_size=cfg.batch_size,
        shuffle=True,
        drop_last=True,
        pin_memory=True
    )

    sc_loader = DataLoader(
        sc_train,
        batch_size=cfg.batch_size,
        shuffle=True,
        drop_last=True,
        pin_memory=True
    )

sc_loader_cycle = cycle(sc_loader)

print("="*60)
print("Datasets Ready")
      
print("="*60)
print(f"SN train :{len(sn_train):,}")
print(f"SC train : {len(sc_train):,}")
print(f"Shared cell types : {len(celltype_to_id)}")
print(f"Balanced sampling : {cfg.balanced_sampling}")
print("="*60)
 # ============================================================
# Cell 06A — Conditional Models
# ============================================================

import torch
import torch.nn as nn
from torch.nn.utils import spectral_norm

# ------------------------------------------------------------
# Residual Block
# ------------------------------------------------------------

class ResBlock(nn.Module):

    def __init__(self, dim, dropout=0.10):

        super().__init__()

        hidden = dim // 2

        self.block = nn.Sequential(

            nn.Linear(dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.SiLU(),

            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),
            nn.SiLU(),

            nn.Linear(hidden, dim),
            nn.BatchNorm1d(dim),

            nn.Dropout(dropout)

        )

        self.attn = nn.Sequential(

            nn.Linear(dim, dim // 8),
            nn.SiLU(),

            nn.Linear(dim // 8, dim),
            nn.Sigmoid()

        )

    def forward(self, x):

        residual = self.block(x)

        residual = residual * self.attn(x)

        return x + residual


# ------------------------------------------------------------
# Conditional Generator
# ------------------------------------------------------------

class Generator(nn.Module):

    def __init__(
        self,
        latent_dim,
        hidden_dim,
        n_celltypes,
        embedding_dim,
        n_blocks=4,
        dropout=0.10
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            n_celltypes,
            embedding_dim
        )

        self.input = nn.Sequential(

            nn.Linear(
                latent_dim + embedding_dim,
                hidden_dim
            ),

            nn.BatchNorm1d(hidden_dim),

            nn.SiLU()

        )

        self.blocks = nn.ModuleList(

            [

                ResBlock(
                    hidden_dim,
                    dropout
                )

                for _ in range(n_blocks)

            ]

        )

        self.output = nn.Linear(
            hidden_dim,
            latent_dim
        )

    def forward(
        self,
        z,
        celltype
    ):

        emb = self.embedding(celltype)

        x = torch.cat(
            [z, emb],
            dim=1
        )

        x = self.input(x)

        for block in self.blocks:

            x = block(x)

        return self.output(x)


# ------------------------------------------------------------
# Conditional Critic
# ------------------------------------------------------------

class Critic(nn.Module):

    def __init__(
        self,
        latent_dim,
        hidden_dim,
        n_celltypes,
        embedding_dim
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            n_celltypes,
            embedding_dim
        )

        self.features = nn.Sequential(

            spectral_norm(

                nn.Linear(

                    latent_dim + embedding_dim,

                    hidden_dim * 4

                )

            ),

            nn.LeakyReLU(
                0.2,
                inplace=True
            ),

            spectral_norm(

                nn.Linear(

                    hidden_dim * 4,

                    hidden_dim * 2

                )

            ),

            nn.LeakyReLU(
                0.2,
                inplace=True
            ),

            spectral_norm(

                nn.Linear(

                    hidden_dim * 2,

                    hidden_dim

                )

            ),

            nn.LeakyReLU(
                0.2,
                inplace=True
            )

        )

        # minibatch statistics
        self.minibatch = nn.Linear(
            hidden_dim,
            16
        )

        self.output = nn.Linear(
            hidden_dim + 16,
            1
        )

    def forward(
        self,
        z,
        celltype
    ):

        emb = self.embedding(celltype)

        x = torch.cat(
            [z, emb],
            dim=1
        )

        h = self.features(x)

        stats = torch.std(
            h,
            dim=0,
            keepdim=True
        ).expand_as(h)

        stats = self.minibatch(stats)

        h = torch.cat(
            [h, stats],
            dim=1
        )

        return self.output(h)


# ------------------------------------------------------------
# Instantiate Models
# ------------------------------------------------------------

N_CELL_TYPES = len(celltype_to_id)

G_sn2sc = Generator(

    latent_dim=cfg.n_latent,

    hidden_dim=cfg.gan_hidden,

    n_celltypes=N_CELL_TYPES,

    embedding_dim=cfg.embedding_dim,

    n_blocks=cfg.gan_res_blocks

).to(device)

G_sc2sn = Generator(

    latent_dim=cfg.n_latent,

    hidden_dim=cfg.gan_hidden,

    n_celltypes=N_CELL_TYPES,

    embedding_dim=cfg.embedding_dim,

    n_blocks=cfg.gan_res_blocks

).to(device)

C_sc = Critic(

    latent_dim=cfg.n_latent,

    hidden_dim=cfg.gan_hidden,

    n_celltypes=N_CELL_TYPES,

    embedding_dim=cfg.embedding_dim

).to(device)

C_sn = Critic(

    latent_dim=cfg.n_latent,

    hidden_dim=cfg.gan_hidden,

    n_celltypes=N_CELL_TYPES,

    embedding_dim=cfg.embedding_dim

).to(device)

print("=" * 60)

print("Conditional GAN Models Initialized")

print("=" * 60)

print(f"Number of cell types : {N_CELL_TYPES}")

print(f"Generator parameters : {sum(p.numel() for p in G_sn2sc.parameters()):,}")

print(f"Critic parameters    : {sum(p.numel() for p in C_sc.parameters()):,}")

print("=" * 60)


# ============================================================

# Cell 06B — GAN Loss Base

# ============================================================

import torch 
import torch.nn as nn 
import numpy as np 
from scipy.linalg import sqrtm

class GANLoss:

    def __init__(self, cfg):

        self.cfg = cfg
        self.l1 = nn.L1Loss()
        self.mse = nn.MSELoss()

    # --------------------------------------------------------
    # Gradient Penalty
    # --------------------------------------------------------

    def gradient_penalty(
        self,
        critic,
        real,
        fake,
        celltype,
        device
    ):

        alpha = torch.rand(real.size(0),1,device=device)

        interp = (
            alpha * real +
            (1-alpha) * fake
        ).requires_grad_(True)

        score = critic(interp, celltype)

        grads = torch.autograd.grad(
            outputs=score,
            inputs=interp,
            grad_outputs=torch.ones_like(score),
            create_graph=True,
            retain_graph=True
        )[0]

        grads = grads.view(grads.size(0), -1)

        gp = ((grads.norm(2, dim=1)-1)**2).mean()

        return gp

    # --------------------------------------------------------
    # Cycle
    # --------------------------------------------------------

    def cycle_loss(self, pred, target):

        return self.l1(pred, target)

    # --------------------------------------------------------
    # Identity
    # --------------------------------------------------------

    def identity_loss(self, pred, target):

        return self.l1(pred, target)

    # --------------------------------------------------------
    # Cell-type Mean Alignment
    # --------------------------------------------------------

    def weighted_celltype_loss(
        self,
        fake,
        fake_ct,
        real,
        real_ct,
        weights
    ):

        loss = 0.0
        total = 0.0

        for ct in torch.unique(fake_ct):

            fm = fake_ct == ct
            rm = real_ct == ct

            if fm.sum() == 0 or rm.sum() == 0:
                continue

            w = weights[ct]

            loss += w * self.mse(
                fake[fm].mean(0),
                real[rm].mean(0)
            )

            total += w

        if total == 0:
            return torch.tensor(
                0.0,
                device=fake.device
            )

        return loss / total

    # --------------------------------------------------------
    # Global Centroid Alignment
    # --------------------------------------------------------

    def centroid_loss(
        self,
        fake,
        celltype,
        centroids
    ):

        return self.mse(
            fake,
            centroids[celltype]
        )

    # --------------------------------------------------------
    # Fréchet Latent Distance
    # --------------------------------------------------------

    @staticmethod
    def calculate_fld(z1, z2):

        mu1 = z1.mean(axis=0)
        mu2 = z2.mean(axis=0)

        s1 = np.cov(z1, rowvar=False)
        s2 = np.cov(z2, rowvar=False)

        cov = sqrtm(s1 @ s2)

        if np.iscomplexobj(cov):
            cov = cov.real

        diff = mu1 - mu2

        return float(
            diff @ diff +
            np.trace(
                s1 + s2 - 2 * cov
            )
        )


# ============================================================

# Cell 06C — Generator Loss

# ============================================================

class GANLoss(GANLoss):

    def generator_loss(

        self,

        G_sn2sc,
        G_sc2sn,

        C_sc,
        C_sn,

        sn_real,
        sc_real,

        sn_ct,
        sc_ct,

        sn_centroids,
        sc_centroids,

        sn_weights,
        sc_weights

    ):

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        fake_sc = G_sn2sc(sn_real, sn_ct)
        fake_sn = G_sc2sn(sc_real, sc_ct)

        # ----------------------------------------------------
        # Adversarial
        # ----------------------------------------------------

        loss_adv = (
            -C_sc(fake_sc, sn_ct).mean()
            -C_sn(fake_sn, sc_ct).mean()
        )

        # ----------------------------------------------------
        # Cycle
        # ----------------------------------------------------

        rec_sn = G_sc2sn(fake_sc, sn_ct)
        rec_sc = G_sn2sc(fake_sn, sc_ct)

        loss_cycle = self.cfg.lambda_cycle * (
            self.cycle_loss(rec_sn, sn_real)
            +
            self.cycle_loss(rec_sc, sc_real)
        )

        # ----------------------------------------------------
        # Identity
        # ----------------------------------------------------

        id_sc = G_sn2sc(sc_real, sc_ct)
        id_sn = G_sc2sn(sn_real, sn_ct)

        loss_identity = self.cfg.lambda_id * (
            self.identity_loss(id_sc, sc_real)
            +
            self.identity_loss(id_sn, sn_real)
        )

        # ----------------------------------------------------
        # Cell-type Loss
        # ----------------------------------------------------

        loss_cell_sn2sc = self.cfg.lambda_cell * (
            self.weighted_celltype_loss(
                fake_sc,
                sn_ct,
                sc_real,
                sc_ct,
                sc_weights
            )
        )

        loss_cell_sc2sn = self.cfg.lambda_cell * (
            self.weighted_celltype_loss(
                fake_sn,
                sc_ct,
                sn_real,
                sn_ct,
                sn_weights
            )
        )

        loss_cell = (
            loss_cell_sn2sc +
            loss_cell_sc2sn
        )

        # ----------------------------------------------------
        # Centroid Loss
        # ----------------------------------------------------

        loss_centroid_sn2sc = self.cfg.lambda_centroid * (
            self.centroid_loss(
                fake_sc,
                sn_ct,
                sc_centroids
            )
        )

        loss_centroid_sc2sn = self.cfg.lambda_centroid * (
            self.centroid_loss(
                fake_sn,
                sc_ct,
                sn_centroids
            )
        )

        loss_centroid = (
            loss_centroid_sn2sc +
            loss_centroid_sc2sn
        )

        # ----------------------------------------------------
        # Total
        # ----------------------------------------------------

        total = (
            loss_adv
            + loss_cycle
            + loss_identity
            + loss_cell
            + loss_centroid
        )

        return {

            "total": total,

            "adv": loss_adv,

            "cycle": loss_cycle,

            "identity": loss_identity,

            "cell": loss_cell,

            "cell_sn2sc": loss_cell_sn2sc,

            "cell_sc2sn": loss_cell_sc2sn,

            "centroid": loss_centroid,

            "centroid_sn2sc": loss_centroid_sn2sc,

            "centroid_sc2sn": loss_centroid_sc2sn,

            "fake_sc": fake_sc,

            "fake_sn": fake_sn

        }
# ============================================================

# Cell 06D — Critic Loss & Optimizers

# ============================================================

class GANLoss(GANLoss):

    # --------------------------------------------------------
    # Critic Loss (WGAN-GP)
    # --------------------------------------------------------

    def critic_loss(
        self,
        critic,
        real,
        fake,
        celltype,
        device
    ):

        d_real = critic(
            real,
            celltype
        )

        d_fake = critic(
            fake,
            celltype
        )

        gp = self.gradient_penalty(
            critic,
            real,
            fake,
            celltype,
            device
        )

        loss = (
            -(d_real.mean() - d_fake.mean())
            +
            self.cfg.lambda_gp * gp
        )

        return loss

# ============================================================

# Instantiate Loss Module

# ============================================================

loss_fn = GANLoss(cfg)

# ============================================================

# Optimizers

# ============================================================

opt_G = optim.Adam(

    list(G_sn2sc.parameters()) +
    list(G_sc2sn.parameters()),

    lr=cfg.lr,

    betas=(0.0, 0.9)

)

opt_D = optim.Adam(

    list(C_sc.parameters()) +
    list(C_sn.parameters()),

    lr=cfg.lr,

    betas=(0.0, 0.9)

)

print("Generator optimizer created.")
print("Critic optimizer created.")

# ============================================================

# Parameter Summary

# ============================================================

g_params = sum( p.numel() for p in G_sn2sc.parameters() if
p.requires_grad )

c_params = sum( p.numel() for p in C_sc.parameters() if p.requires_grad
)


# ============================================================

# Training Setup

# ============================================================

from tqdm.auto import tqdm 
from itertools import cycle 
import json

# ============================================================
# Training History
# ============================================================

history = {

    "epoch": [],

    # Generator losses
    "G_total": [],
    "G_adv": [],
    "G_cycle": [],
    "G_identity": [],

    # Cell losses
    "G_cell": [],
    "G_cell_sn2sc": [],
    "G_cell_sc2sn": [],

    # Centroid losses
    "G_centroid": [],
    "G_centroid_sn2sc": [],
    "G_centroid_sc2sn": [],

    # Critic loss
    "D_loss": [],

    # Evaluation metrics
    "fld": [],
    "best_fld": [],
    "baseline_fld": [],

}

best_fld = float("inf")

fld_baseline = loss_fn.calculate_fld( z_sn_test_np, z_sc_test_np )

print("=" * 60)
print("CycleGAN V2 Training") 
print("=" * 60)
print(f"Baseline FLD : {fld_baseline:.4f}") 
print(f"Epochs :{cfg.gan_epochs}") 
print(f"Batch size : {cfg.batch_size}")
print("=" *60)

sc_loader_cycle = cycle(sc_loader)

for epoch in range(1, cfg.gan_epochs + 1):

    G_sn2sc.train()
    G_sc2sn.train()

    C_sc.train()
    C_sn.train()

    running = {
        "D":0.0,
        "G":0.0,
        "adv":0.0,
        "cycle":0.0,
        "identity":0.0,
        "cell":0.0,
        "cell_sn2sc":0.0,
        "cell_sc2sn":0.0,
        "centroid":0.0,
        "centroid_sn2sc":0.0,
        "centroid_sc2sn":0.0
    }

    n_batches = 0

    for sn_batch in tqdm(sn_loader, leave=False):

        sc_batch = next(sc_loader_cycle)

        sn_real = sn_batch["latent"].to(device)
        sn_label = sn_batch["celltype"].to(device)

        sc_real = sc_batch["latent"].to(device)
        sc_label = sc_batch["celltype"].to(device)
# ============================================================

# Cell 07B — Training Step

# ============================================================

        # ----------------------------------------------------
        # Critic Updates
        # ----------------------------------------------------

        for _ in range(cfg.n_critic):

            opt_D.zero_grad()

            fake_sc = G_sn2sc(
                sn_real,
                sn_label
            ).detach()

            fake_sn = G_sc2sn(
                sc_real,
                sc_label
            ).detach()

            loss_D_sc = loss_fn.critic_loss(
                C_sc,
                sc_real,
                fake_sc,
                sc_label,
                device
            )

            loss_D_sn = loss_fn.critic_loss(
                C_sn,
                sn_real,
                fake_sn,
                sn_label,
                device
            )

            loss_D = loss_D_sc + loss_D_sn

            loss_D.backward()

            opt_D.step()

        # ----------------------------------------------------
        # Generator Update
        # ----------------------------------------------------

        opt_G.zero_grad()

        losses = loss_fn.generator_loss(

            G_sn2sc,
            G_sc2sn,

            C_sc,
            C_sn,

            sn_real,
            sc_real,

            sn_label,
            sc_label,

            sn_centroids,
            sc_centroids,

            sn_weights,
            sc_weights

        )

        losses["total"].backward()

        opt_G.step()

        # ----------------------------------------------------
        # Running Statistics
        # ----------------------------------------------------

        running["D"] += loss_D.item()
        running["G"] += losses["total"].item()
        running["adv"] += losses["adv"].item()
        running["cycle"] += losses["cycle"].item()
        running["identity"] += losses["identity"].item()

        running["cell"] += losses["cell"].item()
        running["cell_sn2sc"] += losses["cell_sn2sc"].item()
        running["cell_sc2sn"] += losses["cell_sc2sn"].item()

        running["centroid"] += losses["centroid"].item()
        running["centroid_sn2sc"] += losses["centroid_sn2sc"].item()
        running["centroid_sc2sn"] += losses["centroid_sc2sn"].item()

        n_batches += 1

    # --------------------------------------------------------
    # Epoch Average
    # --------------------------------------------------------

    for k in running:
        running[k] /= n_batches
# ============================================================

#Evaluation, Checkpointing & History

# ============================================================

    fld = float("nan")
    per_ct_fld = {}

    if epoch == 1 or epoch % cfg.fld_eval_every == 0:

        G_sn2sc.eval()

        translated = []

        with torch.no_grad():

            for start in range(0, len(z_sn_test_np), cfg.batch_size):

                end = min(start + cfg.batch_size, len(z_sn_test_np))

                z_batch = torch.tensor(
                    z_sn_test_np[start:end],
                    dtype=torch.float32,
                    device=device
                )

                ct_batch = torch.tensor(
                    sn_ct_test[start:end],
                    dtype=torch.long,
                    device=device
                )

                translated.append(
                    G_sn2sc(z_batch, ct_batch).cpu().numpy()
                )

        translated = np.concatenate(translated, axis=0)

        fld = loss_fn.calculate_fld(
            translated,
            z_sc_test_np
        )

        for ct in np.unique(sn_ct_test):

            sm = sn_ct_test == ct
            cm = sc_ct_test == ct

            if sm.sum() < 20 or cm.sum() < 20:
                continue

            per_ct_fld[int(ct)] = loss_fn.calculate_fld(
                translated[sm],
                z_sc_test_np[cm]
            )

        if fld < best_fld:

            best_fld = fld

            torch.save(
                {
                    "epoch": epoch,
                    "fld": fld,
                    "G_sn2sc": G_sn2sc.state_dict(),
                    "G_sc2sn": G_sc2sn.state_dict(),
                    "C_sc": C_sc.state_dict(),
                    "C_sn": C_sn.state_dict(),
                    "optimizer_G": opt_G.state_dict(),
                    "optimizer_D": opt_D.state_dict(),
                    "per_ct_fld": per_ct_fld
                },
                os.path.join(
                    cfg.output_dir,
                    "best_gan.pt"
                )
           
         )
        



        # --------------------------------------------------------
        # Save checkpoint every epoch
        # --------------------------------------------------------

        checkpoint_path = os.path.join(
            cfg.output_dir,
            f"gan_epoch_{epoch:03d}.pt"
        )

        torch.save(
            {
                "epoch": epoch,
                "fld": fld,
                "G_sn2sc": G_sn2sc.state_dict(),
                "G_sc2sn": G_sc2sn.state_dict(),
                "C_sc": C_sc.state_dict(),
                "C_sn": C_sn.state_dict(),
                "optimizer_G": opt_G.state_dict(),
                "optimizer_D": opt_D.state_dict(),
                "per_ct_fld": per_ct_fld
            },
            checkpoint_path
        )

        print(f"Saved checkpoint: {checkpoint_path}")
    # ============================================================
    # Save epoch history
    # ============================================================

    history["epoch"].append(epoch)

    # Critic
    history["D_loss"].append(
        running["D"]
    )

    # Generator
    history["G_total"].append(
        running["G"]
    )

    history["G_adv"].append(
        running["adv"]
    )

    history["G_cycle"].append(
        running["cycle"]
    )

    history["G_identity"].append(
        running["identity"]
    )

    # Cell losses
    history["G_cell"].append(
        running["cell"]
    )

    history["G_cell_sn2sc"].append(
        running["cell_sn2sc"]
    )

    history["G_cell_sc2sn"].append(
        running["cell_sc2sn"]
    )

    # Centroid losses
    history["G_centroid"].append(
        running["centroid"]
    )

    history["G_centroid_sn2sc"].append(
        running["centroid_sn2sc"]
    )

    history["G_centroid_sc2sn"].append(
        running["centroid_sc2sn"]
    )

    # FLD
    history["fld"].append(
        fld
    )

    # Best FLD so far
    history["best_fld"].append(
        best_fld
    )

    # Baseline FLD
    history["baseline_fld"].append(
        fld_baseline
    )

    print(
        f"Epoch {epoch:03d}/{cfg.gan_epochs} | "
        f"D={running['D']:.4f} | "
        f"G={running['G']:.4f} | "
        f"FLD={'--' if np.isnan(fld) else f'{fld:.3f}'} | "
        f"Best={best_fld:.3f}"
    )

torch.save( { "epoch": cfg.gan_epochs, "fld": fld, "G_sn2sc":
G_sn2sc.state_dict(), "G_sc2sn": G_sc2sn.state_dict(), "C_sc":
C_sc.state_dict(), "C_sn": C_sn.state_dict(), "optimizer_G":
opt_G.state_dict(), "optimizer_D": opt_D.state_dict() },
os.path.join(cfg.output_dir, "final_gan.pt") )

with open( os.path.join(cfg.output_dir, "gan_history.json"), "w" ) as f:
    json.dump(history, f, indent=2)

with open( os.path.join(cfg.output_dir, "per_celltype_fld.json"), "w" ) as f: 
    json.dump(per_ct_fld, f, indent=2)

print("=" * 60) 
print("Training Complete") 
print(f"Baseline FLD : {fld_baseline:.3f}") 
print(f"Best FLD : {best_fld:.3f}") 
print("=" * 60)
# ============================================================

# Training Curves

# ============================================================

import matplotlib.pyplot as plt 
import numpy as np

epochs = history["epoch"]

fig, axes = plt.subplots(2,2, figsize=(15,10))



axes[0,0].plot(epochs, history["D_loss"], lw=2)
axes[0,0].set_title("Critic Loss") 
axes[0,0].grid(alpha=0.3)



axes[0,1].plot(epochs, history["G_total"], label="Total", lw=2)
axes[0,1].plot(epochs, history["G_adv"], label="Adv")
axes[0,1].plot(epochs, history["G_cycle"], label="Cycle")
axes[0,1].plot(epochs, history["G_identity"], label="Identity")
axes[0,1].plot(epochs, history["G_cell"], label="Cell")
axes[0,1].plot(epochs, history["G_centroid"], label="Centroid")
axes[0,1].legend(fontsize=8) 
axes[0,1].set_title("Generator Losses")
axes[0,1].grid(alpha=0.3)



fld_epochs=[e for e,f in zip(epochs,history["fld"]) if not np.isnan(f)]
fld_vals=[f for f in history["fld"] if not np.isnan(f)]

if fld_vals: axes[1,0].plot(fld_epochs,fld_vals,"o",lw=2)
axes[1,0].axhline(fld_baseline,color="red",ls="-",label="Baseline")
axes[1,0].legend()

axes[1,0].set_title("FLD") 
axes[1,0].grid(alpha=0.3)



if fld_vals: imp=[fld_baseline-f for f in fld_vals]
axes[1,1].plot(fld_epochs,imp,"g-o")
axes[1,1].axhline(0,color="black",ls=":")

axes[1,1].set_title("FLD Improvement") 
axes[1,1].grid(alpha=0.3)

plt.tight_layout()

plt.savefig( os.path.join( cfg.output_dir, "training_curves.png" ),
dpi=200, bbox_inches="tight" )

plt.show()

# Decoding of test sets

In [ ]:
# ============================================================

# Cell 09 — Translate SN Test Latents

# ============================================================


print("=" *60) 
print("Loading best generator") 
print("="*60)

ckpt = torch.load( os.path.join(cfg.output_dir, "best_gan.pt"),
map_location=device )

G_sn2sc.load_state_dict(ckpt["G_sn2sc"])
G_sn2sc.eval()



translated = []

with torch.no_grad():

    for start in tqdm(range(0, len(z_sn_test_np), cfg.batch_size)):

        end = min(start + cfg.batch_size, len(z_sn_test_np))

        z_batch = torch.tensor(
            z_sn_test_np[start:end],
            dtype=torch.float32,
            device=device
        )

        ct_batch = torch.tensor(
            sn_ct_test[start:end],
            dtype=torch.long,
            device=device
        )

        pred = G_sn2sc(
            z_batch,
            ct_batch
        )

        translated.append(pred.cpu().numpy())

z_sn_translated = np.concatenate(translated, axis=0)

print("Translated shape:", z_sn_translated.shape)

np.savez_compressed( os.path.join( cfg.output_dir,
"sn_test_translated_latent.npz" ), z=z_sn_translated,
cell_type=sn_cell_types[idx_sn_test], cell_type_id=sn_ct_test,
batch_idx=sn_batch_idx[idx_sn_test],
dataset_id=sn_dataset_ids[idx_sn_test],
log_library=log_lib_sn[idx_sn_test] )

stats = {}

for ct in np.unique(sn_ct_test):

    mask = sn_ct_test == ct

    stats[int(ct)] = {
        "n_cells": int(mask.sum()),
        "latent_mean": z_sn_translated[mask].mean(0).tolist(),
        "latent_std": z_sn_translated[mask].std(0).tolist()
    }

with open( os.path.join( cfg.output_dir, "translation_statistics.json"
), "w" ) as f: json.dump(stats, f, indent=2)

print("Saved translated latent space and statistics.")
# ============================================================

# Cell 10 — Decode Translated Latents with SCVI

# ============================================================
print("Loading SCVI decoder...")

adata_sc = sc.read_h5ad(cfg.sc_data)

model_sc = scvi.model.SCVI.load(
    cfg.sc_model,
    adata=adata_sc
)

model_sc.module.eval()
print("="*60)
print("Decoding translated latent space") 
print("="*60)



latent_file = os.path.join(cfg.output_dir,
"sn_test_translated_latent.npz") 
lat = np.load(latent_file, allow_pickle=True)

z_sn_translated = lat["z"] 
sn_ct_test = lat["cell_type_id"]

decode_library = sc_ct_library[sn_ct_test] 
decode_batch = sc_ct_batch[sn_ct_test]

decoded_counts = np.zeros( (len(z_sn_translated), cfg.n_genes),
dtype=np.float32 )

model_sc.module.eval()

for start in tqdm(range(0, len(z_sn_translated), cfg.decode_chunk)):

    end = min(start + cfg.decode_chunk, len(z_sn_translated))

    z_chunk = torch.tensor(
        z_sn_translated[start:end],
        dtype=torch.float32,
        device=device
    )

    lib_chunk = torch.tensor(
        decode_library[start:end],
        dtype=torch.float32,
        device=device
    ).unsqueeze(1)

    batch_chunk = torch.tensor(
        decode_batch[start:end],
        dtype=torch.long,
        device=device
    ).unsqueeze(1)

    with torch.no_grad():

        out = model_sc.module.generative(
            z=z_chunk,
            library=lib_chunk,
            batch_index=batch_chunk
        )

        decoded_counts[start:end] = out["px"].mu.cpu().numpy()

np.savez_compressed( os.path.join( cfg.output_dir,
"sn_test_translated_raw_counts.npz" ), counts=decoded_counts,
cell_type_id=sn_ct_test )

print("Saved decoded raw-count matrix.")

pred_pseudobulk = {} 
pred_counts = {}

for ct in np.unique(sn_ct_test):

    mask = sn_ct_test == ct

    pred_pseudobulk[int(ct)] = (
        decoded_counts[mask]
        .mean(axis=0)
        .astype(np.float32)
    )

    pred_counts[int(ct)] = int(mask.sum())

np.savez_compressed( os.path.join( cfg.output_dir, "pred_pseudobulk.npz"
), **{str(k):v for k,v in pred_pseudobulk.items()} )

print(f"Decoded {len(pred_pseudobulk)} cell types.")

# ————————-

# Sanity check

# ————————-

idx = idx_sc_test[:1000]

with torch.no_grad():

    sanity = model_sc.module.generative(

        z=torch.tensor(
            z_sc[idx],
            dtype=torch.float32,
            device=device
        ),

        library=torch.tensor(
            log_lib_sc[idx],
            dtype=torch.float32,
            device=device
        ).unsqueeze(1),

        batch_index=torch.tensor(
            sc_batch_idx[idx],
            dtype=torch.long,
            device=device
        ).unsqueeze(1)

    )["px"].mu.cpu().numpy()

print("Sanity check") 
print("Mean :", sanity.mean()) 
print("Std :",
sanity.std()) 
print("Range:", sanity.min(), sanity.max())